# 1. tokenizer1:

Create

## 1. Classes

1. Part 1: Class `CharTokenizer`
   - 1.1 encode
   - 1.2 decode
2. Part 2: Class `BPETokenizer`
   - 2.1 _get_pairs
   - 2.2 _merge_pair
   - 2.3 train
   - 2.4 encode
   - 2.5 decode
   - 2.6 vocab_size
   - 2.7 token_to_str

## 2. Functiones

1. Part 3: `compression_ratio`
   - Calculate token compression ratio relative to raw UTF-8 byte count
   - Handle empty strings gracefully by returning 0.0 when byte length is zero

2. Part 4: `vocabulary_stats`
   - Compute and display usage statistics across a list of text corpora

3. demo_char_tokenizer
4. demo_bpe_training
5. demo_encode_decode
6. demo_tiktoken_comparison
7. demo_vocabulary_analysis


Import Packages

In [1]:
from collections import Counter
from typing import List, Dict, Tuple, Any, Optional


# 1. Classes

1. Part 1: Class `CharTokenizer`
   - 1.1 encode
   - 1.2 decode
2. Part 2: Class `BPETokenizer`
   - 2.1 _get_pairs
   - 2.2 _merge_pair
   - 2.3 train
   - 2.4 encode
   - 2.5 decode
   - 2.6 vocab_size
   - 2.7 token_to_str



### Part 1: `CharTokenizer`

In [2]:
#  Part 1:---------------------------------------------
class CharTokenizer:
    """A simple character-level tokenizer mapping ASCII/Unicode characters to integer values."""

    def encode(self, text: str) -> List[int]:
        """Convert an input string into a list of integer character codes.

        Args:
            text (str): Input text to encode.

        Returns:
            List[int]: List of character integer codes.
        """
        # TODO: Map each character in the string to its integer character code representation
        return [ord(char) for char in text] # Unicode Code Point

    
    def decode(self, tokens: List[int]) -> str:
        """Convert a list of integer character codes back into a string.

        Args:
            tokens (List[int]): List of integer character codes.

        Returns:
            str: Decoded text string.
        """
        # TODO: Map each integer code back to its corresponding character and combine them
        return "".join(chr(token) for token in tokens)


In [3]:
# manual test
# 1. CharTokenizer -> encode

print("/1/".center(40, '-'))
sample_1 = "hello word"
print(f"main text--------\n{sample_1}")
print(f"main text--------\n{list(sample_1)}")
print(f"encode-----------\n{[ord(char) for char in sample_1]}")
print("-------------------")
print(f"len text:    {len(list(sample_1))}")
print(f"len encode:  {len([ord(char) for char in sample_1])}")

# 1. CharTokenizer -> decode
print("/2/".center(40, '-'))

sample_2 = [ord(char) for char in sample_1]
print(f"decode-----------\n{"".join(chr(token) for token in sample_2)}")


------------------/1/-------------------
main text--------
hello word
main text--------
['h', 'e', 'l', 'l', 'o', ' ', 'w', 'o', 'r', 'd']
encode-----------
[104, 101, 108, 108, 111, 32, 119, 111, 114, 100]
-------------------
len text:    10
len encode:  10
------------------/2/-------------------
decode-----------
hello word


In [4]:
# test class CharTokenizer
print("/test_1/".center(40, '-'))
sample_1 = "hello word"
Test_CharTokenizer = CharTokenizer()
test_encode = Test_CharTokenizer.encode(sample_1)
print(f"main Text---------\n{sample_1}")
print(f"Text encode-------\n{test_encode}")

print("/test_2/".center(40, '-'))
test_decode = Test_CharTokenizer.decode(test_encode)
print(f"Text decode-------\n{test_decode}")

----------------/test_1/----------------
main Text---------
hello word
Text encode-------
[104, 101, 108, 108, 111, 32, 119, 111, 114, 100]
----------------/test_2/----------------
Text decode-------
hello word


### Part 2: `BPETokenizer`

In [5]:
# Part 2:---------------------------------------------
class BPETokenizer:
    """Byte-Pair Encoding (BPE) Tokenizer implementation starting from 256 base byte tokens."""

    def __init__(self) -> None:
        """Initialize vocabulary and merge rules storage."""
        self.merges: Dict[Tuple[int, int], int] = {}
        self.vocab: Dict[int, bytes] = {}

    def _get_pairs(self, tokens: List[int]) -> Counter:
        """Count frequencies of adjacent token pairs in a sequence.

        Args:
            tokens (List[int]): List of current integer tokens. Shape: (seq_len,)

        Returns:
            Counter: Mapping of (token_a, token_b) tuples to their occurrence counts.
        """
        # TODO: Count frequency of adjacent pairs across the token sequence
        return Counter(zip(tokens, tokens[1:]))

    def _merge_pair(
        self, tokens: List[int], pair: Tuple[int, int], new_token: int
    ) -> List[int]:
        """Replace non-overlapping occurrences of a target adjacent pair with a new merged token ID.

        Args:
            tokens (List[int]): Current token sequence. Shape: (seq_len,)
            pair (Tuple[int, int]): Target pair of token IDs to replace.
            new_token (int): New token ID to insert for the target pair.

        Returns:
            List[int]: Updated sequence with non-overlapping target pair merged.
        """
        # TODO: Iterate through sequence and substitute target pair occurrences with new token without overlapping
        #-----Byte Pair Encoding------
        # Whenever we see a repeating pattern, we replace it with a new number
        # EX: [101, 104 ,105, 106, 108, 101, 104] -> [225, 105, 106, 107, 225]
        # [101, 104] repited and change to 225
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                new_tokens.append(new_token)
                i += 2  # jump from token
            else:
                new_tokens.append(tokens[i])
                i += 1  # Add token and go to next
        return new_tokens

    def train(self, text: str, num_merges: int) -> "BPETokenizer":
        """Train BPE vocabulary starting from 256 base bytes and learn merge rules from text.

        Args:
            text (str): Raw text corpus for training.
            num_merges (int): Total number of BPE merge operations to execute.

        Returns:
            BPETokenizer: Self instance after training.
        """
        # TODO: Initialize 256 base byte tokens in vocab, compute pair frequencies, and iteratively merge top pair
        #--------The core engine of the BPE algorithm (Greedy)---------
        # 1. base vocablory with 256 byit (0 to 255)
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.merges = {}

        # 2. Change text to encode
        tokens = list(text.encode("utf-8"))

        # Main loop
        for i in range(num_merges):
            # Calculating pair frequencies
            stats = self._get_pairs(tokens)
            if not stats:
                break # Stop if no pairs remain

            # Finding the most frequent pair
            best_pair = stats.most_common(1)[0][0]
            new_token = 256 + i # New token ID (starts from 256)

            # Record merge in rules and update vocabulary
            self.merges[best_pair] = new_token
            self.vocab[new_token] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

            # Replace the pair in the token list for the next iteration
            tokens = self._merge_pair(tokens, best_pair, new_token)

        # Method Chaining or Fluent Interface:
        # 1. can Fluent API
        # 2. Library standards
        return self

    def encode(self, text: str) -> List[int]:
        """Encode string text into BPE token IDs using learned merge rules.

        Args:
            text (str): Input text string.

        Returns:
            List[int]: Encoded list of BPE token IDs.
        """
        # TODO: Convert text into raw UTF-8 bytes and iteratively apply learned merge rules in order
        if not text:
            return []

        # 1. Change text to basic byit
        tokens = list(text.encode("utf-8"))

        # 2. Apply learned merge rules in the same order they were recorded in train
        for pair, new_token in self.merges.items():
            tokens = self._merge_pair(tokens, pair, new_token)

        return tokens

    def decode(self, tokens: List[int]) -> str:
        """Decode a list of BPE token IDs back into a text string.

        Args:
            tokens (List[int]): List of BPE token IDs.

        Returns:
            str: Reconstructed text string.
        """
        # TODO: Lookup byte sequences for tokens, concatenate them, and decode UTF-8 bytes into text
        all_bytes = b"".join(self.vocab[token] for token in tokens)
        return all_bytes.decode("utf-8")

    def vocab_size(self) -> int:
        """Get current vocabulary size.

        Returns:
            int: Total number of unique tokens in the vocabulary.
        """
        # TODO: Return total number of items in vocabulary
        return len(self.vocab)

    def token_to_str(self, token_id: int) -> str:
        """Convert a single token ID into its string representation for visualization.

        Args:
            token_id (int): Token ID to inspect.

        Returns:
            str: String representation of the token bytes.
        """
        # TODO: Fetch byte mapping for token ID and decode into printable string format
        # ---------Visualization-----------
        token_bytes = self.vocab.get(token_id)
        if token_bytes is None:
            return ""
        return token_bytes.decode("utf-8", errors="replace")


In [6]:
# manual test
# BPETokenizer -> _get_pairs

print("/1/".center(40, '-'))
sample_2 = "hello word I am mohsen mohebbi"
test_encode_2 = [ord(char) for char in sample_2]
test__get_pairs = Counter(zip(test_encode_2, test_encode_2[1:]))
print(f"main text--------\n{sample_2}")
print(f"encode-----------\n{test_encode_2}")
print(f"_get_pairs-------\n{test__get_pairs}")
print(f"Description------")
print(sample_2[0],sample_2[1])
print(f"encode: {(104, 101)} count repeated: {test__get_pairs[(104, 101)]}")

# BPETokenizer -> encode to utf-8
print("/2/".center(40, '-'))
test_encode_utf8 = sample_2.encode('utf-8')
print(f"encode by utf-8 - b in the first is type (bytes)\n {test_encode_utf8}")
# if change to list, see code
print(list(test_encode_utf8))



------------------/1/-------------------
main text--------
hello word I am mohsen mohebbi
encode-----------
[104, 101, 108, 108, 111, 32, 119, 111, 114, 100, 32, 73, 32, 97, 109, 32, 109, 111, 104, 115, 101, 110, 32, 109, 111, 104, 101, 98, 98, 105]
_get_pairs-------
Counter({(104, 101): 2, (32, 109): 2, (109, 111): 2, (111, 104): 2, (101, 108): 1, (108, 108): 1, (108, 111): 1, (111, 32): 1, (32, 119): 1, (119, 111): 1, (111, 114): 1, (114, 100): 1, (100, 32): 1, (32, 73): 1, (73, 32): 1, (32, 97): 1, (97, 109): 1, (109, 32): 1, (104, 115): 1, (115, 101): 1, (101, 110): 1, (110, 32): 1, (101, 98): 1, (98, 98): 1, (98, 105): 1})
Description------
h e
encode: (104, 101) count repeated: 2
------------------/2/-------------------
encode by utf-8 - b in the first is type (bytes)
 b'hello word I am mohsen mohebbi'
[104, 101, 108, 108, 111, 32, 119, 111, 114, 100, 32, 73, 32, 97, 109, 32, 109, 111, 104, 115, 101, 110, 32, 109, 111, 104, 101, 98, 98, 105]


# 2. Functiones

1. Part 3: `compression_ratio`
   - Calculate token compression ratio relative to raw UTF-8 byte count
   - Handle empty strings gracefully by returning 0.0 when byte length is zero

2. Part 4: `vocabulary_stats`
   - Compute and display usage statistics across a list of text corpora

3. demo_char_tokenizer
4. demo_bpe_training
5. demo_encode_decode
6. demo_tiktoken_comparison
7. demo_vocabulary_analysis



In [7]:
# Part 3----------------------------------------
def compression_ratio(tokenizer: Any, text: str) -> float:
    """Calculate token compression ratio relative to raw UTF-8 byte count.

    Handle empty strings gracefully by returning 0.0 when byte length is zero.

    Args:
        tokenizer (Any): Tokenizer instance providing encode().
        text (str): Input text string.

    Returns:
        float: Ratio of encoded token count to raw byte length (0.0 if empty).
    """
    # TODO: Calculate encoded token count relative to raw UTF-8 byte length, returning 0.0 for empty input
    # This function checks how many tokens the text is converted into after tokenization, relative to its raw UTF-8 byte count.

    byte_length = len(text.encode("utf-8"))

    if byte_length == 0:
        return 0.0

    token_count = len(tokenizer.encode(text))

    return token_count / byte_length


In [8]:
# manual test
print("/compression_ratio/".center(40, '-'))
sample_3 = "hello word I am mohsen mohebbi"

test_tokenizer = BPETokenizer() # class

test_tokenizer.train(sample_3, num_merges=7) # 7 paie token merge
test_ratio = compression_ratio(test_tokenizer, sample_3)

print(f"sampel text---\n{sample_3}")
print(f"Byte length: {len(sample_3.encode('utf-8'))}")
print(f"Token count: {len(test_tokenizer.encode(sample_3))}")
print(f"Compression ratio: {test_ratio} ---manual {len(test_tokenizer.encode(sample_3)) / len(sample_3.encode('utf-8'))}")


----------/compression_ratio/-----------
sampel text---
hello word I am mohsen mohebbi
Byte length: 30
Token count: 20
Compression ratio: 0.6666666666666666 ---manual 0.6666666666666666


In [9]:
# part 4------------------------------------------------------
def vocabulary_stats(tokenizer: Any, texts: List[str]) -> None:
    """Compute and display usage statistics across a list of text corpora.

    Args:
        tokenizer (Any): Tokenizer instance providing encode(), vocab_size(), and token_to_str().
        texts (List[str]): List of text strings to analyze.

    Returns:
        None
    """
    # TODO: Compute token frequency metrics, average token lengths per word, and output summary stats
    token_freq = Counter()
    total_tokens = 0
    total_words = 0

    for text in texts:
        tokens = tokenizer.encode(text)
        token_freq.update(tokens)
        total_tokens += len(tokens)
        total_words += len(text.split())

    vocab_size = tokenizer.vocab_size()
    avg_tokens_per_word = total_tokens / total_words if total_words > 0 else 0.0

    print(f"Vocabulary Size: {vocab_size}") # 255 + merge_num
    print(f"Total Tokens: {total_tokens}") # sum tokens
    print(f"Total Words: {total_words}") # count vocab
    print(f"Average Tokens Per Word: {avg_tokens_per_word:.2f}")
    print("Top 10 Frequent Tokens:")
    for token_id, count in token_freq.most_common(10):
        print(f"  Token {token_id} ({tokenizer.token_to_str(token_id)!r}): {count}")

In [10]:
# manual test
print("/vocabulary_stats - 1/".center(40, '-'))
sample_3 = "hello word I am mohsen mohebbi"
sample_4 = "I am learning Mini Project"
test_text = [sample_3, sample_4]
test_token_freq = Counter()
test_tokenizer = BPETokenizer() # class

for text in test_text:
    print(text)

    tokens = test_tokenizer.encode(text)
    print(tokens)

    test_token_freq.update(tokens)
    print(test_token_freq)
    print("-\/\/\/\/\/\/\-")

print("/vocabulary_stats_ 2/".center(40, '-'))

test_tokenizer = BPETokenizer() # class
test_tokenizer.train(" ".join(test_text), num_merges=10)
vocabulary_stats(test_tokenizer, test_text)



---------/vocabulary_stats - 1/---------
hello word I am mohsen mohebbi
[104, 101, 108, 108, 111, 32, 119, 111, 114, 100, 32, 73, 32, 97, 109, 32, 109, 111, 104, 115, 101, 110, 32, 109, 111, 104, 101, 98, 98, 105]
Counter({32: 5, 111: 4, 104: 3, 101: 3, 109: 3, 108: 2, 98: 2, 119: 1, 114: 1, 100: 1, 73: 1, 97: 1, 115: 1, 110: 1, 105: 1})
-\/\/\/\/\/\/\-
I am learning Mini Project
[73, 32, 97, 109, 32, 108, 101, 97, 114, 110, 105, 110, 103, 32, 77, 105, 110, 105, 32, 80, 114, 111, 106, 101, 99, 116]
Counter({32: 9, 101: 5, 111: 5, 109: 4, 110: 4, 105: 4, 104: 3, 108: 3, 114: 3, 97: 3, 73: 2, 98: 2, 119: 1, 100: 1, 115: 1, 103: 1, 77: 1, 80: 1, 106: 1, 99: 1, 116: 1})
-\/\/\/\/\/\/\-
---------/vocabulary_stats_ 2/----------
Vocabulary Size: 266
Total Tokens: 43
Total Words: 11
Average Tokens Per Word: 3.91
Top 10 Frequent Tokens:
  Token 32 (' '): 6
  Token 111 ('o'): 3
  Token 114 ('r'): 3
  Token 101 ('e'): 3
  Token 262 ('mo'): 2
  Token 110 ('n'): 2
  Token 98 ('b'): 2
  Token 105 ('

<>:17: SyntaxWarning: invalid escape sequence '\/'
<>:17: SyntaxWarning: invalid escape sequence '\/'
C:\Users\mohse\AppData\Local\Temp\ipykernel_24308\3962661861.py:17: SyntaxWarning: invalid escape sequence '\/'
  print("-\/\/\/\/\/\/\-")


In [11]:
# [KEEP_IMPLEMENTATION]
def demo_char_tokenizer() -> None:
    """Demonstrate basic character-level tokenizer operations."""
    print("=" * 60)
    print("STEP 1: Character-Level Tokenizer")
    print("=" * 60)

    ct = CharTokenizer()

    texts = ["hello", "Hello, world!", "GPT-4"]
    for text in texts:
        encoded = ct.encode(text)
        decoded = ct.decode(encoded)
        print(f"  '{text}' -> {encoded}")
        print(f"  Roundtrip: {'PASS' if decoded == text else 'FAIL'}")
        print(f"  Tokens: {len(encoded)}")
        print()


# [KEEP_IMPLEMENTATION]
def demo_bpe_training() -> Tuple[BPETokenizer, str]:
    """Demonstrate BPE training process on a sample text corpus."""
    print("=" * 60)
    print("STEP 2: BPE Training")
    print("=" * 60)

    corpus = (
        "The cat sat on the mat. The cat ate the rat. "
        "The dog sat on the log. The dog ate the frog. "
        "Natural language processing is the study of how computers "
        "understand and generate human language. "
        "Tokenization is the first step in any NLP pipeline. "
        "Language models read tokens, not words. "
        "The tokenizer converts text into a sequence of integers. "
        "Each integer maps to a subword in the vocabulary."
    )

    tokenizer = BPETokenizer()
    tokenizer.train(corpus, num_merges=50)

    print(f"\nVocabulary size after training: {tokenizer.vocab_size()}")
    print(f"Number of merges learned: {len(tokenizer.merges)}")

    return tokenizer, corpus


# [KEEP_IMPLEMENTATION]
def demo_encode_decode(tokenizer: BPETokenizer) -> None:
    """Demonstrate BPE encoding and decoding functionality across sentences."""
    print("\n" + "=" * 60)
    print("STEP 3: Encode and Decode")
    print("=" * 60)

    test_sentences = [
        "The cat sat on the mat.",
        "Natural language processing",
        "tokenization pipeline",
        "unhappiness",
        "The dog ate the frog.",
    ]

    for sentence in test_sentences:
        encoded = tokenizer.encode(sentence)
        decoded = tokenizer.decode(encoded)
        raw_bytes = len(sentence.encode("utf-8"))
        ratio = len(encoded) / raw_bytes
        roundtrip = "PASS" if decoded == sentence else "FAIL"
        print(f"\n  '{sentence}'")
        print(f"  Encoded: {encoded[:15]}{'...' if len(encoded) > 15 else ''}")
        print(f"  Tokens: {len(encoded)} (from {raw_bytes} bytes)")
        print(f"  Compression ratio: {ratio:.2f}")
        print(f"  Roundtrip: {roundtrip}")


# [KEEP_IMPLEMENTATION]
def demo_tiktoken_comparison(tokenizer: BPETokenizer) -> None:
    """Compare custom BPE tokenizer outputs with OpenAI's tiktoken implementation."""
    print("\n" + "=" * 60)
    print("STEP 4: Compare with tiktoken")
    print("=" * 60)

    try:
        import tiktoken
    except ImportError:
        print("  tiktoken not installed. Run: pip install tiktoken")
        return

    enc = tiktoken.get_encoding("cl100k_base")

    texts = [
        "The cat sat on the mat.",
        "unhappiness",
        "Hello, world!",
        "def fibonacci(n): return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)",
        "Geschwindigkeitsbegrenzung",
    ]

    for text in texts:
        our_tokens = tokenizer.encode(text)
        tk_tokens = enc.encode(text)
        tk_pieces = [enc.decode([t]) for t in tk_tokens]
        print(f"\n  '{text}'")
        print(f"  Our BPE:  {len(our_tokens)} tokens")
        print(f"  tiktoken: {len(tk_tokens)} tokens -> {tk_pieces}")
        ratio = len(our_tokens) / len(tk_tokens) if len(tk_tokens) > 0 else 0
        print(f"  Ours / tiktoken: {ratio:.1f}x")


# [KEEP_IMPLEMENTATION]
def demo_vocabulary_analysis(tokenizer: BPETokenizer, corpus: str) -> None:
    """Demonstrate vocabulary statistic evaluation and compression ratio checks."""
    print("\n" + "=" * 60)
    print("STEP 5: Vocabulary Analysis")
    print("=" * 60)

    test_texts = [
        corpus,
        "The quick brown fox jumps over the lazy dog.",
        "Machine learning is a subset of artificial intelligence.",
        "Python is the most popular language for data science.",
    ]

    vocabulary_stats(tokenizer, test_texts)

    print(f"\nCompression ratios:")
    for text in test_texts[:3]:
        preview = text[:50] + "..." if len(text) > 50 else text
        ratio = compression_ratio(tokenizer, text)
        print(f"  {ratio:.2f} -- '{preview}'")



In [12]:
# ===== UNIT TESTS =====


def test_char_tokenizer() -> None:
    ct = CharTokenizer()

    # Standard functionality test
    sample = "Hello World!"
    encoded = ct.encode(sample)
    decoded = ct.decode(encoded)
    assert isinstance(encoded, list), "CharTokenizer.encode should return a list."
    assert (
        len(encoded) == len(sample)
    ), f"CharTokenizer output length mismatch. Expected {len(sample)}, got {len(encoded)}"
    assert (
        decoded == sample
    ), f"Roundtrip decoding failed. Expected '{sample}', got '{decoded}'"

    # Edge Case: Unicode Characters
    unicode_sample = "Hello 🌍!"
    u_encoded = ct.encode(unicode_sample)
    u_decoded = ct.decode(u_encoded)
    assert (
        u_decoded == unicode_sample
    ), f"Unicode decoding failed. Expected '{unicode_sample}', got '{u_decoded}'"

    # Edge Case: Empty String
    empty_encoded = ct.encode("")
    assert empty_encoded == [], "Encoding empty string should yield an empty list."
    assert ct.decode([]) == "", "Decoding empty list should yield an empty string."


def test_bpe_tokenizer_get_pairs() -> None:
    bpe = BPETokenizer()

    tokens = [1, 2, 1, 2, 3]
    pairs = bpe._get_pairs(tokens)
    assert pairs[(1, 2)] == 2, "Pair counting incorrect for pair (1, 2)."
    assert pairs[(2, 3)] == 1, "Pair counting incorrect for pair (2, 3)."

    # Edge Case: Single element / empty list
    assert (
        len(bpe._get_pairs([1])) == 0
    ), "Single element input should yield 0 pairs."
    assert len(bpe._get_pairs([])) == 0, "Empty list input should yield 0 pairs."


def test_bpe_tokenizer_merge_pair() -> None:
    bpe = BPETokenizer()

    tokens = [1, 2, 1, 2, 3]
    merged = bpe._merge_pair(tokens, (1, 2), 99)
    assert merged == [
        99,
        99,
        3,
    ], f"Merge output mismatch. Expected [99, 99, 3], got {merged}"

    # Non-overlapping merge edge case test
    overlapping_tokens = [1, 1, 1]
    non_overlap_merged = bpe._merge_pair(overlapping_tokens, (1, 1), 99)
    assert non_overlap_merged == [
        99,
        1,
    ], f"Merge must be non-overlapping. Expected [99, 1], got {non_overlap_merged}"

    # Edge Case: Pair not present / Sequence length 1
    assert bpe._merge_pair([1, 2, 3], (4, 5), 99) == [
        1,
        2,
        3,
    ], "Merging non-existent pair should return unchanged list."
    assert bpe._merge_pair([1], (1, 2), 99) == [
        1
    ], "Merging sequence of length 1 should return unchanged list."


def test_bpe_tokenizer_train_and_encode_decode() -> None:
    bpe = BPETokenizer()
    corpus = "aaabcaaab"

    bpe.train(corpus, num_merges=2)

    # Base byte requirement: Must initialize 256 base bytes + 2 merges = 258 vocab size
    assert (
        bpe.vocab_size() == 258
    ), f"BPE must start with 256 base bytes. Expected vocab_size 258 after 2 merges, got {bpe.vocab_size()}"
    assert (
        len(bpe.merges) == 2
    ), f"Learned merges count mismatch. Expected 2, got {len(bpe.merges)}"

    encoded = bpe.encode(corpus)
    decoded = bpe.decode(encoded)

    assert (
        len(encoded) < len(corpus.encode("utf-8"))
    ), "BPE compression failed to reduce token length."
    assert (
        decoded == corpus
    ), f"Roundtrip decoding mismatch. Expected '{corpus}', got '{decoded}'"

    # Edge Case: Unseen character at encoding
    unseen = "z"
    unseen_encoded = bpe.encode(unseen)
    assert bpe.decode(unseen_encoded) == unseen, "Failed to encode/decode unseen character correctly."


def test_bpe_tokenizer_token_to_str() -> None:
    bpe = BPETokenizer()
    bpe.vocab = {0: b"a", 256: b"ab"}

    res_a = bpe.token_to_str(0)
    res_ab = bpe.token_to_str(256)
    res_unknown = bpe.token_to_str(999)

    assert res_a == "a", f"Expected 'a', got '{res_a}'"
    assert res_ab == "ab", f"Expected 'ab', got '{res_ab}'"
    assert isinstance(res_unknown, str), "token_to_str should return string even for unknown token ID."


def test_compression_ratio() -> None:
    ct = CharTokenizer()
    text = "Test string"
    ratio = compression_ratio(ct, text)

    assert isinstance(ratio, float), "Compression ratio must return a float value."
    assert (
        0.0 <= ratio <= 2.0
    ), f"Compression ratio out of reasonable bounds: {ratio}"

    # Edge Case: Empty String (Division by zero check)
    empty_ratio = compression_ratio(ct, "")
    assert empty_ratio == 0.0, f"Expected 0.0 for empty string input, got {empty_ratio}"


def test_vocabulary_stats() -> None:
    class DummyTokenizer:
        def encode(self, text: str) -> List[int]:
            return [1, 2, 3]

        def vocab_size(self) -> int:
            return 10

        def token_to_str(self, token_id: int) -> str:
            return f"tok_{token_id}"

    dt = DummyTokenizer()
    # vocabulary_stats prints output and shouldn't crash
    try:
        vocabulary_stats(dt, ["hello world", "test corpus"])
    except Exception as e:
        assert False, f"vocabulary_stats raised an unexpected exception: {e}"


def run_all_tests() -> None:
    """Run hidden unit tests across student functions."""
    print("Running Unit Tests...")
    test_char_tokenizer()
    test_bpe_tokenizer_get_pairs()
    test_bpe_tokenizer_merge_pair()
    test_bpe_tokenizer_train_and_encode_decode()
    test_bpe_tokenizer_token_to_str()
    test_compression_ratio()
    test_vocabulary_stats()
    print("All unit tests passed successfully!\n")


def run_all_demos() -> None:
    """Run step-by-step demonstration functions."""
    demo_char_tokenizer()
    tokenizer, corpus = demo_bpe_training()
    demo_encode_decode(tokenizer)
    demo_tiktoken_comparison(tokenizer)
    demo_vocabulary_analysis(tokenizer, corpus)


if __name__ == "__main__":
    # Unit tests and pipeline demos are explicitly separated
    run_all_tests()
    run_all_demos()

Running Unit Tests...
Vocabulary Size: 10
Total Tokens: 6
Total Words: 4
Average Tokens Per Word: 1.50
Top 10 Frequent Tokens:
  Token 1 ('tok_1'): 2
  Token 2 ('tok_2'): 2
  Token 3 ('tok_3'): 2
All unit tests passed successfully!

STEP 1: Character-Level Tokenizer
  'hello' -> [104, 101, 108, 108, 111]
  Roundtrip: PASS
  Tokens: 5

  'Hello, world!' -> [72, 101, 108, 108, 111, 44, 32, 119, 111, 114, 108, 100, 33]
  Roundtrip: PASS
  Tokens: 13

  'GPT-4' -> [71, 80, 84, 45, 52]
  Roundtrip: PASS
  Tokens: 5

STEP 2: BPE Training

Vocabulary size after training: 306
Number of merges learned: 50

STEP 3: Encode and Decode

  'The cat sat on the mat.'
  Encoded: [264, 99, 258, 284, 109, 258, 46]
  Tokens: 7 (from 23 bytes)
  Compression ratio: 0.30
  Roundtrip: PASS

  'Natural language processing'
  Encoded: [78, 258, 117, 114, 97, 108, 290, 256, 112, 114, 291, 101, 115, 115, 262]...
  Tokens: 16 (from 27 bytes)
  Compression ratio: 0.59
  Roundtrip: PASS

  'tokenization pipeline'
  